# P053 — Colab Simulation Runner (v7A)**All fixes applied:**1. GradScaler `init_scale=1024` (prevents float16 overflow with FocalLoss on T4)2. NaN batches **skipped** (not crashed) — GradScaler adapts scale automatically3. Console logging reduced to **warnings only** (file logs still capture everything)4. Failed training **does not block** future retrain attempts (staleness gate reset)**Output per day** (clean, 1 line each):```  Day  1 | 2026-02-20 |  5,000,000 rows |  213.0 MB |    28.5s | —  Day 30 | 2026-03-21 |  5,000,000 rows |  213.0 MB | 17340.5s | RETRAIN_TRIGGERED```**During training** (every epoch):```   1   0.0234   0.0198   0.0312   0.0423   0.001000  530.2s *```**Runs:** fast (0 epochs, ~1 min) then full (50 epochs, ~5 hrs on T4)**Drive outputs:** `MyDrive/P053_results/v7A/{fast,full}/`

In [ ]:
# CELL 1: Clone + Install + Verifyimport os, shutil, subprocessREPO = "https://github.com/AIML-Engineering-Lab/053_dram_memory_yield_mlops.git"DIR = "/content/053_memory_yield_predictor"try:    from google.colab import userdata    tok = userdata.get('GITHUB_TOKEN')except Exception:    tok = Noneurl = REPO.replace("https://", f"https://{tok}@") if tok else REPOif os.path.exists(DIR):    shutil.rmtree(DIR)r = subprocess.run(["git", "clone", url, DIR], capture_output=True, text=True)if r.returncode != 0:    print(r.stderr)    raise RuntimeError("Clone failed. Set GITHUB_TOKEN in Colab Secrets.")os.chdir(DIR)# Verify ALL v7A fixes presentwith open('src/train.py') as f:    train_src = f.read()with open('src/simulation_logger.py') as f:    logger_src = f.read()with open('src/run_simulation.py') as f:    sim_src = f.read()checks = [    ('init_scale=1024' in train_src, 'GradScaler init_scale fix'),    ('nan_batches' in train_src, 'NaN recovery in train_one_epoch'),    ('P053_LOG_VERBOSE' in logger_src, 'Quiet console logging'),    ('training_ok' in sim_src, 'Staleness reset on failure'),]for ok, name in checks:    status = '\u2705' if ok else '\u274c MISSING'    print(f'  {status} {name}')    assert ok, f'{name} — push latest code!'subprocess.run(["pip", "install", "-q", "mlflow", "boto3", "pyarrow", "pandas",                "numpy", "scikit-learn", "xgboost", "lightgbm"],               capture_output=True)print('\u2705 All v7A fixes verified + deps installed')

In [ ]:
# CELL 2: AWS (optional) + GPU checkimport os, torchtry:    from google.colab import userdata    os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')    os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')    print("\u2705 AWS credentials loaded")except Exception:    print("\u26a0\ufe0f No AWS credentials. Drive backup only.")os.environ.update({'AWS_DEFAULT_REGION': 'us-west-2', 'S3_BUCKET': 'p053-mlflow-artifacts',                    'COMPUTE_BACKEND': 'colab', 'MODEL_PARAMS': '317000', 'SIMULATION_SCALE': 'phase2'})if not torch.cuda.is_available():    raise RuntimeError("No GPU! Runtime > Change runtime type > T4 GPU")gpu = torch.cuda.get_device_name(0)vram = torch.cuda.get_device_properties(0).total_memory / 1e9cc = torch.cuda.get_device_capability(0)print(f"\u2705 {gpu} ({vram:.1f} GB, cc {cc[0]}.{cc[1]})")bs = 4096 if cc[0] >= 8 else 2048print(f"   batch_size={bs}, {'bfloat16' if cc[0]>=8 else 'float16+GradScaler(init_scale=1024)'}")

In [ ]:
# CELL 3: Mount Drive + Copy datafrom google.colab import drivefrom pathlib import Pathimport shutildrive.mount('/content/drive')dest = '/content/053_memory_yield_predictor/data/preprocessed_full.npz'Path(dest).parent.mkdir(parents=True, exist_ok=True)for src in ['/content/drive/MyDrive/P053_data/preprocessed_full.npz',            '/content/drive/MyDrive/preprocessed_full.npz']:    if Path(src).exists():        shutil.copy2(src, dest)        mb = Path(dest).stat().st_size / 1e6        print(f'\u2705 Copied {mb:.0f} MB from {src}')        assert mb > 2000, f'Too small ({mb:.0f} MB)'        breakelse:    raise FileNotFoundError('preprocessed_full.npz not found on Drive')

In [ ]:
# CELL 4: Run fast + full with LIVE outputimport json, os, shutil, subprocess, sys, timefrom pathlib import PathDIR = '/content/053_memory_yield_predictor'DRIVE = '/content/drive/MyDrive/P053_results/v7A'Path(DRIVE).mkdir(parents=True, exist_ok=True)RUNS = [    {'name': 'fast', 'rows': 100_000, 'epochs': 0},    {'name': 'full', 'rows': 5_000_000, 'epochs': 50},]results = []def run_with_live_output(cmd, cwd):    env = os.environ.copy()    env['PYTHONUNBUFFERED'] = '1'    proc = subprocess.Popen(        cmd, cwd=cwd, env=env,        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,        text=True, bufsize=1,    )    for line in proc.stdout:        print(line, end='', flush=True)    return proc.wait()for cfg in RUNS:    name, rows, epochs = cfg['name'], cfg['rows'], cfg['epochs']    print(f"\n{'='*70}")    print(f"STARTING {name.upper()} | {rows:,} rows/day | {epochs} epochs")    if epochs > 0:        print(f"  Expected: ~5 hrs on T4 (early stop ~epoch 33, ~530s/epoch)")    print('=' * 70, flush=True)    ckpt = Path(DIR) / 'data' / 'simulation_checkpoint.json'    ckpt.unlink(missing_ok=True)    cmd = [sys.executable, '-u', '-m', 'src.run_simulation',           '--rows', str(rows), '--backend', 'colab', '--checkpoint',           '--sim-retrain-epochs', str(epochs)]    t0 = time.time()    retcode = run_with_live_output(cmd, cwd=DIR)    elapsed = (time.time() - t0) / 60    if retcode != 0:        print(f'\u274c {name} FAILED (exit {retcode})', flush=True)        results.append({'name': name, 'elapsed_min': elapsed, 'status': 'FAILED'})        continue    # Backup to Drive    bk = Path(DRIVE) / name    bk.mkdir(parents=True, exist_ok=True)    for p in ['data/simulation_timeline.json', 'data/simulation_checkpoint.json', 'mlflow.db']:        s = Path(DIR) / p        if s.exists():            dn = f'mlflow_{name}.db' if s.name == 'mlflow.db' else s.name            shutil.copy2(s, bk / dn)    for bm in (Path(DIR) / 'data').glob('benchmark_*.json'):        shutil.copy2(bm, bk / bm.name)    for folder in ['src/artifacts', 'data/drift_reports', 'mlruns', 'data/production', 'data/logs']:        src_d = Path(DIR) / folder        if src_d.exists():            dst = bk / Path(folder).name            if dst.exists(): shutil.rmtree(dst)            shutil.copytree(src_d, dst)    # Parse timeline    tl_path = Path(DIR) / 'data' / 'simulation_timeline.json'    retrains = total_days = 0    train_min = 0.0    if tl_path.exists():        tl = json.loads(tl_path.read_text())        retrains = len(tl.get('retrain_events', []))        total_days = tl.get('total_days', 0)        for d in tl.get('days', []):            if any('RETRAIN' in e for e in d.get('events', [])):                train_min += d.get('elapsed_sec', 0) / 60    results.append({        'name': name, 'rows': rows, 'epochs': epochs,        'elapsed_min': round(elapsed, 1), 'retrains': retrains,        'days': total_days, 'gpu_train_min': round(train_min, 1),        'backup': str(bk), 'status': 'OK',    })    print(f"\n\u2705 {name.upper()} done | {total_days} days | {retrains} retrains | "          f"{elapsed:.1f} min | GPU train: {train_min:.1f} min", flush=True)    print(f"   Saved: {bk}")# Save summaryfor p in [Path(DIR)/'data'/'colab_run_summary.json', Path(DRIVE)/'run_summary.json']:    p.write_text(json.dumps(results, indent=2))print(f"\nSummary: {DRIVE}/run_summary.json", flush=True)

In [ ]:
# CELL 5: Verify resultsimport jsonfrom pathlib import Paths = Path('/content/drive/MyDrive/P053_results/v7A/run_summary.json')if not s.exists():    print('No summary yet. Run Cell 4 first.'); raise SystemExitsummary = json.loads(s.read_text())print('=' * 70)print('RESULTS')print('=' * 70)total = 0.0for r in summary:    total += r['elapsed_min']    ok = '\u2705' if r['status'] == 'OK' else '\u274c'    print(f"{ok} {r['name']:<6} {r.get('days', 0):>2}d  {r.get('retrains', 0)} retrains  "          f"{r['elapsed_min']:>6.1f} min  gpu={r.get('gpu_train_min', 0):.0f} min")print('-' * 70)print(f"Total: {total:.1f} min ({total/60:.1f} hrs)")# Check full run trainingfull = [r for r in summary if r['name'] == 'full']if full and full[0].get('gpu_train_min', 0) > 60:    print(f"\n\u2705 REAL GPU TRAINING: {full[0]['gpu_train_min']:.0f} min")elif full and full[0].get('epochs', 0) > 0:    print(f"\n\u26a0\ufe0f  Training time looks short ({full[0].get('gpu_train_min', 0):.0f} min)")# Check benchmarkfor b in sorted(Path('/content/drive/MyDrive/P053_results/v7A/full').glob('benchmark_*.json'),                key=lambda f: f.stat().st_mtime, reverse=True):    bm = json.loads(b.read_text())    print(f"\nBenchmark: {b.name}")    print(f"  GPU: {bm.get('gpu_name')} | batch: {bm.get('batch_size')}")    print(f"  Epochs: {bm.get('epochs_run')} (best: {bm.get('best_epoch')})")    print(f"  Time: {bm.get('total_train_time_min', 0):.1f} min")    if 'results' in bm and 'val' in bm['results']:        print(f"  Val AUC-PR: {bm['results']['val']['auc_pr']:.4f}")    break# Check model filearts = Path('/content/drive/MyDrive/P053_results/v7A/full/artifacts')if arts.exists():    pts = list(arts.glob('*.pt'))    print(f"\n\u2705 Models: {[p.name for p in pts]}" if pts else "\n\u26a0\ufe0f No .pt files")

---## What to expect**Fast run (~1 min):** 40 days, 0 training epochs, just data generation + drift check.Console output is clean — 1 line per day + warnings only.**Full run (~5 hrs on T4):**- Days 1-29: data gen + drift check (~30s/day each)- Day 30: drift triggers retrain → 50 epochs of GPU training (~530s/epoch)- Days 31-40: continue with new model- Day 39: simulated bad deploy + rollbackIf NaN batches occur during training, you'll see:```  [NaN] batch 129/4883 — skipped (total=1, scale=512)```This is NORMAL — GradScaler adapts. Only crashes if >50% batches are NaN.After completion, copy `MyDrive/P053_results/v7A/full/` to workspace, then:```bashpython -m src.post_simulation_update```---*P053 — AIML Engineering Lab*